# NB02: NMDC Environmental-Microbe Census

**Goal**: Inventory all NMDC multi-omics data with linked environmental parameters. Build the sample -> abiotic features -> taxonomy -> metabolomics linkage table.

**Requires**: BERDL JupyterHub (Spark access)

**Key questions**:
1. How many NMDC samples have abiotic environmental measurements (pH, temp, nutrients)?
2. What ecosystem types are represented and at what scale?
3. What taxonomic profiling methods are available and how do they link to samples?
4. What metabolomics/proteomics/transcriptomics data exists with environmental context?

**Outputs**: `data/nmdc_linkage_summary.csv`, `data/nmdc_abiotic_coverage.csv`, `data/nmdc_sample_ecosystem.csv`

In [ ]:
spark = get_spark_session()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("../data")
FIG_DIR = Path("../figures")
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 150, "figure.figsize": (12, 6)})

## 1. NMDC Database Survey

In [ ]:
# Survey all NMDC tables
nmdc_tables = spark.sql("SHOW TABLES IN nmdc_arkin").toPandas()
print(f"Total NMDC tables: {len(nmdc_tables)}")

# Get row counts for key tables
key_tables = [
    "abiotic_features", "study_table", "metabolomics_gold",
    "proteomics_gold", "lipidomics_gold", "metatranscriptomics_gold",
    "centrifuge_gold", "gottcha_gold", "kraken_gold",
    "annotation_terms_unified", "trait_unified",
    "taxonomy_dim", "embedding_metadata"
]

table_stats = []
for t in key_tables:
    try:
        count = spark.sql(f"SELECT COUNT(*) AS n FROM nmdc_arkin.{t}").collect()[0]["n"]
        table_stats.append({"table": t, "rows": count})
    except Exception as e:
        table_stats.append({"table": t, "rows": f"ERROR: {e}"})

table_stats_df = pd.DataFrame(table_stats)
print("\n=== NMDC Key Table Inventory ===")
print(table_stats_df.to_string(index=False))

## 2. Abiotic Environmental Parameters

Which samples have pH, temperature, nutrients, and metal measurements?

In [ ]:
# Load abiotic features
abiotic = spark.sql("SELECT * FROM nmdc_arkin.abiotic_features").toPandas()
print(f"Total samples with abiotic data: {len(abiotic)}")

# Check which columns have non-zero data
param_cols = [c for c in abiotic.columns if c.startswith("annotations_")]
abiotic_coverage = {}
for col in param_cols:
    param_name = col.replace("annotations_", "").replace("_has_numeric_value", "").replace("_has_", "_")
    n_nonzero = (abiotic[col] != 0).sum()
    n_nonnull = abiotic[col].notna().sum()
    abiotic_coverage[param_name] = {
        "n_samples_nonzero": n_nonzero,
        "n_samples_total": n_nonnull,
        "coverage_pct": n_nonzero / len(abiotic) * 100 if len(abiotic) > 0 else 0
    }

abiotic_cov_df = pd.DataFrame(abiotic_coverage).T.sort_values("n_samples_nonzero", ascending=False)
print("\n=== Abiotic Parameter Coverage (non-zero values) ===")
print(abiotic_cov_df.to_string())

abiotic_cov_df.to_csv(DATA_DIR / "nmdc_abiotic_coverage.csv")
print(f"\nSaved to {DATA_DIR / 'nmdc_abiotic_coverage.csv'}")

In [ ]:
# Figure: Abiotic parameter coverage
fig, ax = plt.subplots(figsize=(12, 6))
plot_data = abiotic_cov_df[abiotic_cov_df["n_samples_nonzero"] > 0].sort_values("n_samples_nonzero")
ax.barh(range(len(plot_data)), plot_data["n_samples_nonzero"], color="#3498db")
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(plot_data.index, fontsize=9)
ax.set_xlabel("Number of samples with measurement")
ax.set_title("NMDC Abiotic Parameter Coverage (non-zero values)")
plt.tight_layout()
plt.savefig(FIG_DIR / "nmdc_abiotic_coverage.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. NMDC Study & Ecosystem Coverage

In [ ]:
# Study-level summary
studies = spark.sql("SELECT * FROM nmdc_arkin.study_table").toPandas()
print(f"Total NMDC studies: {len(studies)}")
print(f"\nStudy columns: {studies.columns.tolist()}")
print(f"\nFirst 10 studies:")
print(studies.head(10).to_string())

## 4. Taxonomic Profiling Data

Check what metagenome-derived taxonomy is available (Centrifuge, GOTTCHA, Kraken).

In [ ]:
# Check taxonomic profiling tables
for tool in ["centrifuge_gold", "gottcha_gold", "kraken_gold"]:
    try:
        schema = spark.sql(f"DESCRIBE nmdc_arkin.{tool}").toPandas()
        count = spark.sql(f"SELECT COUNT(*) AS n FROM nmdc_arkin.{tool}").collect()[0]["n"]
        n_samples = spark.sql(
            f"SELECT COUNT(DISTINCT file_id) AS n FROM nmdc_arkin.{tool}"
        ).collect()[0]["n"]
        print(f"\n{tool}: {count:,} rows, {n_samples} samples")
        print(f"  Columns: {schema['col_name'].tolist()[:10]}...")
    except Exception as e:
        print(f"\n{tool}: ERROR - {e}")

## 5. Metabolomics & Multi-omics Coverage

In [ ]:
# Count metabolomics, proteomics, lipidomics, transcriptomics samples
omics_summary = {}
for omics in ["metabolomics_gold", "proteomics_gold", "lipidomics_gold", "metatranscriptomics_gold"]:
    try:
        count = spark.sql(f"SELECT COUNT(*) AS n FROM nmdc_arkin.{omics}").collect()[0]["n"]
        n_files = spark.sql(
            f"SELECT COUNT(DISTINCT file_id) AS n FROM nmdc_arkin.{omics}"
        ).collect()[0]["n"]
        omics_summary[omics] = {"total_records": count, "unique_files": n_files}
    except Exception as e:
        omics_summary[omics] = {"total_records": f"ERROR", "unique_files": 0}

omics_df = pd.DataFrame(omics_summary).T
print("=== NMDC Multi-omics Scale ===")
print(omics_df.to_string())

## 6. Sample-File Linkage and Overlap

Determine how many samples have abiotic measurements AND taxonomic/metabolomic data.

In [ ]:
# Check sample-file lookup table to understand how samples map to omics files
sample_files = spark.sql("""
    SELECT * FROM nmdc_arkin.sample_file_lookup
    LIMIT 20
""").toPandas()
print(f"Sample-file lookup columns: {sample_files.columns.tolist()}")
print(f"\nSample:")
print(sample_files.head(10).to_string())

# Count unique samples in lookup
total_samples = spark.sql(
    "SELECT COUNT(DISTINCT sample_id) AS n FROM nmdc_arkin.sample_file_lookup"
).collect()[0]["n"]
print(f"\nTotal unique samples in file lookup: {total_samples}")

In [ ]:
# Identify overlap: samples with abiotic data AND at least one omics type
abiotic_samples = set(abiotic[abiotic.drop(columns=["sample_id"]).any(axis=1)]["sample_id"])
print(f"Samples with ANY non-zero abiotic measurement: {len(abiotic_samples)}")

# Check which abiotic samples also appear in the sample_file_lookup
# (meaning they have associated omics data)
all_file_samples = spark.sql(
    "SELECT DISTINCT sample_id FROM nmdc_arkin.sample_file_lookup"
).toPandas()["sample_id"].tolist()
all_file_samples_set = set(all_file_samples)

abiotic_with_omics = abiotic_samples & all_file_samples_set
print(f"Abiotic samples also in file lookup (have omics): {len(abiotic_with_omics)}")
print(f"Total file-linked samples: {len(all_file_samples_set)}")
print(f"Overlap rate: {len(abiotic_with_omics)/max(len(abiotic_samples),1)*100:.1f}%")

## 7. NMDC Embeddings & Traits

Check the pre-computed embeddings and trait data that could enhance environmental predictions.

In [ ]:
# Check embedding and trait tables
for table in ["embedding_metadata", "trait_unified", "trait_sources", "trait_taxonomy_mapping"]:
    try:
        schema = spark.sql(f"DESCRIBE nmdc_arkin.{table}").toPandas()
        count = spark.sql(f"SELECT COUNT(*) AS n FROM nmdc_arkin.{table}").collect()[0]["n"]
        print(f"\n{table}: {count:,} rows")
        print(f"  Columns: {schema['col_name'].tolist()}")
    except Exception as e:
        print(f"\n{table}: ERROR - {e}")

## 8. Summary

Compile NMDC environmental-microbe linkage summary for integration with ENIGMA census (NB01).

In [ ]:
# Compile NMDC linkage summary
nmdc_summary = {
    "NMDC Studies": len(studies),
    "Total samples in abiotic table": len(abiotic),
    "Samples with non-zero abiotic data": len(abiotic_samples),
    "Abiotic parameters measured": len([c for c in abiotic_cov_df.index
                                         if abiotic_cov_df.loc[c, "n_samples_nonzero"] > 0]),
    "Samples with omics files": len(all_file_samples_set),
    "Samples with BOTH abiotic + omics": len(abiotic_with_omics),
}

summary_df = pd.DataFrame([{"metric": k, "value": v} for k, v in nmdc_summary.items()])
summary_df.to_csv(DATA_DIR / "nmdc_linkage_summary.csv", index=False)

print("=== NMDC Environmental-Microbe Linkage Summary ===\n")
for k, v in nmdc_summary.items():
    print(f"  {k:50s} {v:>8,}")

print(f"\nSaved to {DATA_DIR / 'nmdc_linkage_summary.csv'}")